# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "What Predicts Health?" (ML Appendix — Random Forest feature importance for health_score)

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health_score, and — to its credit — explicitly flags the risk itself: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."



*   Where does the label come from? health_score is defined earlier in the paper as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — a formula, not an independently observed outcome. Two of the three top "predictors" (impressions, position) are literally addends in the score being predicted.List item

* Does the validation design carry the claim? The paper reports an 80/20 holdout split but doesn't specify whether it's grouped by client or brand. Given this project's own finding — that a naive random split (45/47 clients overlapping) produced a different validation number than a grouped split on structurally similar data — it's worth asking whether the paper's 80/20 split controls for client-level correlation the same way, since 57 brands could plausibly cluster in feature space the way our 47 clients did.

*   Constructive framing: the paper already does the hardest part right — naming the circularity itself rather than hiding it. The one addition that would strengthen this further: report feature importance for a residual target (health_score with impressions/position's own point-contributions subtracted out), so the top predictors describe genuinely independent signal, not partially the scoring formula reflecting itself.

Finding 2: "Captured Traffic Value" (Finding #9 — clicks × CPC as a value proxy)

The paper reports $253.5K in "captured click-equivalent value" across the portfolio, explicitly choosing clicks × CPC over the "unsafe" impressions × CPC ($73.0M) — a genuinely good methodological correction the paper makes itself.

### Where does the label/benchmark come from?

CPC is described as "a cost-per-click benchmark" but its source isn't specified in the methodology section — is it observed ad-auction data per keyword, an industry-average estimate, or a client-reported figure? This matters because the entire $253.5K figure is a linear function of this one input, and the paper's own Confounding Variables section notes "revenue tracking covers 8 of 57 clients" — meaning the dollar figure is being validated against real revenue for only a small fraction of the underlying data.

### Does the validation design carry the claim?

This finding is presented as a direct aggregate calculation, not a model needing train/val validation in the usual sense — appropriately so, since it's arithmetic on observed clicks, not a prediction. The open methodology question is narrower: is the CPC figure itself validated against the 8 clients with real revenue tracking, or applied uniformly across all 57 as an untested assumption?

### Constructive framing

The paper's own instinct (rejecting impressions × CPC as "unsafe") shows exactly the right caution — extending that same caution to the CPC input's own provenance, and reporting the $253.5K figure alongside a confidence range or explicitly separating the 8-client-verified subset from the 49-client-extrapolated remainder, would make an already-careful finding fully defensible end to end.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Lane 3's clustering question is unsupervised and cross-sectional — no label, single time window — but pages from the same client are not independent observations: a client's content plausibly shares a CMS template, an editorial word-count convention, and a consistent GSC tracking setup. A random row split risks letting a client's "house style" appear in both train and validation, inflating apparent generalization. A client-grouped split is the honest design for this reason; the comparison below makes that concrete rather than asserted.

Week 5's model used a client-grouped split (GroupShuffleSplit on client_hash_id) from the start. This section builds the "before" we never actually ran at the time — a naive, ungrouped split — and compares it side by side with the honest, grouped version.

Before (naive random split): validation silhouette 0.2962, Davies-Bouldin 1.0752, with 45 of 47 clients appearing in both train and validation sets — meaning the validation set is not a genuine held-out test of client generalization.

After (honest client-grouped split): validation silhouette 0.3568, Davies-Bouldin 0.9087, with 0 client overlap — confirmed by direct set intersection, not assumed.

The two splits produce measurably different results from identical data and an identical model, differing only in split design. Notably, the naive split's silhouette is lower, not higher, than the grouped split's — the opposite of what we initially expected from a memorization-inflation effect. This is worth reporting honestly rather than smoothing over: split design measurably changes the number, and only the client-grouped version's near-total client separation actually earns the label "validation." The grouped-split result (0.3568) is the one carried forward as this model's honest performance figure, on methodological grounds — because it genuinely tests unseen clients — not because it happens to be the higher number.

All claims above are observed, measured comparisons within this dataset and split design — not causal statements about why one split "performs better," and not a claim that either number would hold under a different sample or time period.

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- Rebuild the same w03/w05 feature frame, unchanged ---
raw = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id
     AND a.content_hash_id = d.content_hash_id
    ORDER BY a.client_hash_id, a.content_hash_id
""").df()

df = raw.copy()
df['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days
df['avg_position_missing_or_zero'] = (df['gsc_avg_position'] == 0).astype(int)
df['avg_position_clean'] = df['gsc_avg_position'].replace(0, np.nan)
avg_position_median = df.loc[df['avg_position_clean'].notna(), 'avg_position_clean'].median()
df['avg_position_clean'] = df['avg_position_clean'].fillna(avg_position_median)
df['log_gsc_impressions'] = np.log1p(df['gsc_impressions'])
df['log_gsc_clicks'] = np.log1p(df['gsc_clicks'])
df['word_count_missing'] = df['word_count'].isna().astype(int)
word_count_median = df['word_count'].median()
df['word_count'] = df['word_count'].fillna(word_count_median)

MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                   'content_age_days', 'word_count',
                   'avg_position_missing_or_zero', 'word_count_missing']
FINAL_K = 4

def fit_and_score(train_df, val_df, split_name):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[MODEL_FEATURES])
    X_val = scaler.transform(val_df[MODEL_FEATURES])

    km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=30)
    train_labels = km.fit_predict(X_train)
    val_labels = km.predict(X_val)

    val_sil = silhouette_score(X_val, val_labels, sample_size=20000, random_state=42)
    val_db = davies_bouldin_score(X_val, val_labels)

    n_train_clients = train_df['client_hash_id'].nunique()
    n_val_clients = val_df['client_hash_id'].nunique()
    client_overlap = len(set(train_df['client_hash_id']) & set(val_df['client_hash_id']))

    return {
        'split': split_name,
        'train_rows': len(train_df), 'val_rows': len(val_df),
        'train_clients': n_train_clients, 'val_clients': n_val_clients,
        'client_overlap': client_overlap,
        'val_silhouette': round(val_sil, 4),
        'val_davies_bouldin': round(val_db, 4),
    }

# --- BEFORE: naive random row split (no grouping) ---
train_naive, val_naive = train_test_split(df, test_size=0.25, random_state=42)
before_result = fit_and_score(train_naive, val_naive, 'BEFORE — naive random row split')

# --- AFTER: honest client-grouped split (matches w05) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id'].values))
train_grouped = df.iloc[train_idx].reset_index(drop=True)
val_grouped = df.iloc[val_idx].reset_index(drop=True)
after_result = fit_and_score(train_grouped, val_grouped, 'AFTER — honest client-grouped split')

comparison = pd.DataFrame([before_result, after_result])
print("Before/after split comparison:")
print(comparison.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before/after split comparison:
                              split  train_rows  val_rows  train_clients  val_clients  client_overlap  val_silhouette  val_davies_bouldin
    BEFORE — naive random row split      132553     44185             47           45              45          0.2962              1.0752
AFTER — honest client-grouped split      133474     43264             35           12               0          0.3568              0.9087


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Static field check. The final, w05-committed 7-feature set (log_gsc_impressions, avg_position_clean, log_gsc_clicks, content_age_days, word_count, avg_position_missing_or_zero, word_count_missing) contains zero forbidden fields, confirmed by both an exact-match check against the standing forbidden-fields list (health_score, priority_score, action_type, baseline_flag, in_w04_baseline_queue, etc.) and a pattern-based substring check. The two missingness flags contain no forbidden substrings and are legitimate data-availability indicators, not leaked targets — consistent with every prior leakage check across w03, w04, and w05.

Deliberate-leak demonstration. Repeating w03's leak trap on the final feature set: an honest K-Means fit on the full 176,738-row dataset (not the train/val split — this check evaluates the feature set's integrity independent of split design) produced a silhouette of 0.2953. Appending a one-hot encoded version of the model's own cluster output as a "feature," then re-scoring against the same (unchanged) cluster assignment, pushed the silhouette to 0.3549 — a +0.0596 (~20% relative) jump, from an input that is mathematically the answer restated, not new information. The leaked column was removed; 0.2953 is the only number reported as valid for this specific check.

In [ ]:
# --- Part 1: Static forbidden-field check on the FINAL feature set ---
forbidden_fields = {
    "health_score", "priority_score", "action_type", "recommended_action",
    "recommendation", "action_label", "reason_code", "cluster", "archetype",
    "label", "target", "is_declining_label", "future", "post", "outcome",
    "conversion", "optimization_eligible_date", "last_optimized_date", "is_deleted",
    "baseline_flag", "baseline_score", "in_w04_baseline_queue",
}

FINAL_MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                          'content_age_days', 'word_count',
                          'avg_position_missing_or_zero', 'word_count_missing']

leaked_in_final_features = forbidden_fields.intersection(set(FINAL_MODEL_FEATURES))
print("Forbidden fields in the FINAL feature set:", leaked_in_final_features or "NONE — clean")
print("Final feature set:", FINAL_MODEL_FEATURES)

# Pattern-based check, same discipline as w04/w05 (catches near-matches, not just exact names)
forbidden_patterns = ["label", "target", "future", "post", "outcome", "recommend",
                       "action_type", "health", "priority", "cluster", "archetype",
                       "baseline", "flag_x", "trend"]
pattern_hits = [f for f in FINAL_MODEL_FEATURES if any(p in f.lower() for p in forbidden_patterns)]
print("\nPattern-based hits on final features:", pattern_hits or "NONE — clean")
print("(Note: 'avg_position_missing_or_zero' and 'word_count_missing' contain no forbidden")
print("substrings — they are legitimate missingness flags, not leaked target fields.)")

# --- Part 2: The deliberate-leak demonstration, repeated on the FINAL feature set ---
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_honest = df[FINAL_MODEL_FEATURES].copy()
X_honest_scaled = StandardScaler().fit_transform(X_honest)

km_honest = KMeans(n_clusters=4, random_state=42, n_init=30)
labels_honest = km_honest.fit_predict(X_honest_scaled)
score_honest = silhouette_score(X_honest_scaled, labels_honest, sample_size=20000, random_state=42)
print(f"\nHONEST silhouette (final 7-feature set): {score_honest:.4f}")

# THE TRAP: one-hot encode the model's own cluster output, append it as a "feature"
leak_onehot = OneHotEncoder(sparse_output=False).fit_transform(labels_honest.reshape(-1, 1))
X_leaky = np.hstack([X_honest_scaled, leak_onehot])
score_leaky = silhouette_score(X_leaky, labels_honest, sample_size=20000, random_state=42)
print(f"LEAKY silhouette (cluster-derived proxy appended): {score_leaky:.4f}")
print(f"Jump: {score_honest:.4f} -> {score_leaky:.4f}  (+{score_leaky - score_honest:.4f})")

print(f"\nLeak removed. Honest silhouette {score_honest:.4f} is the only number reported as valid.")

Forbidden fields in the FINAL feature set: NONE — clean
Final feature set: ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks', 'content_age_days', 'word_count', 'avg_position_missing_or_zero', 'word_count_missing']

Pattern-based hits on final features: NONE — clean
(Note: 'avg_position_missing_or_zero' and 'word_count_missing' contain no forbidden
substrings — they are legitimate missingness flags, not leaked target fields.)

HONEST silhouette (final 7-feature set): 0.2953
LEAKY silhouette (cluster-derived proxy appended): 0.3549
Jump: 0.2953 -> 0.3549  (+0.0596)

Leak removed. Honest silhouette 0.2953 is the only number reported as valid.


Note on which silhouette number is "the" model result: this 0.2953 figure is distinct from Section 2's reported validation silhouette of 0.3568 — that number came from the client-grouped validation split specifically (the honest, held-out generalization test), while 0.2953 here comes from fitting on the entire dataset at once (appropriate for testing feature-set integrity, not for claiming validated performance). The model's honest, reportable performance figure remains 0.3568 (Section 2's grouped-split validation number) — 0.2953 is a leakage-audit artifact, not a competing headline metric, and should not be quoted interchangeably with it.

What this confirms: the same leakage mechanism identified in w03 — a feature derived from the model's own output inflates apparent structure — still holds on the exact feature set actually shipped in the final model, not just an earlier candidate version. This reinforces why in_w04_baseline_queue was correctly used only as a post-hoc overlay diagnostic in w05, never as a clustering input.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Per the notebook's hint — "take your own boldest sentence and rewrite it in safe language" — the clearest candidate from this project's own history is the original section title from w02: "Why ML beats a fixed rule here."

**Original claim:**

  *"ML beats a fixed rule here... Real archetypes don't respect box-shaped boundaries — a rule-based approach would need a human to hand-pick thresholds..."*

Checked against the claim ladder from writing-honest-claims: "beats" is a comparative-superiority claim stated as settled fact, without a controlled comparison between the two approaches on the same task and metric. What we actually had at that point was reasoning about why K-Means is structurally better suited to a discovery task than a hand-written rule — a methodological argument, not a measured head-to-head result. "Beats" implies a resolved contest; the evidence available was closer to "this method's mechanics fit the problem shape better," which sits lower on the claim ladder.

This exact overreach was caught during review at the time and corrected to:

**Revised claim (already adopted in w02):**

  *"ML is better suited here because the goal is to discover unknown behavioral archetypes from multiple interacting signals... A fixed rule remains the right tool when the decision boundary is already known and interpretability is the top priority — but that's not this task."*

Why the revision is honest, mapped explicitly to the claim ladder:



*   "is better suited... because the goal is..." — this is a reasoned methodological claim, not a comparative performance claim. It doesn't say ML produced a better number than a rule would have; it says the two methods answer different question shapes, and this lane's question shape favors K-Means.

*   The revision keeps the counter-case explicit ("a fixed rule remains the right tool when...") rather than implying universal superiority — matching the skill file's own instruction to frame findings constructively, not as an absolute win.
No banned words present (no "proves," "causes," "beats" removed entirely).

*  No banned words present (no "proves," "causes," "beats" removed entirely).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.